# 02 — Backfill Histórico Amplo (2 Anos: Bitcoin + Ethereum)

Reconstrói **2 anos (730 dias)** de dados horários para Bitcoin e Ethereum em paralelo com a coleta da Solana em andamento no `02_Backfill.ipynb`.

- **Nova pasta isolada:** Salva em `dados_cripto/historico_2anos/` sem tocar no backfill original.
- **Bitcoin:** ~105.000 blocos históricos coletados via `mempool.space` com checkpoints periódicos a cada 50 chamadas.
- **Ethereum:** ~5.256.000 blocos coletados em lotes de 1.024 blocos via `eth_feeHistory` em RPCs com histórico pós-Merge.
- **Mercado & Sentimento:** Candles de 1h da Binance e série diária do Fear & Greed integrados na mesma grade horária.

In [1]:
import os, time, json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# ============ CONFIGURAÇÃO DE 2 ANOS ============
DIAS = 730                    # 2 anos de histórico
REFAZER = False               # True reinicia os checkpoints do zero
BTC_TX_VBYTES = 141           # P2WPKH: 1 entrada e 2 saídas
ETH_GAS_TRANSFERENCIA = 21_000

if os.path.exists("/content") and "PASTA_DADOS" not in globals():
    PASTA_DADOS = Path("/content/dados_cripto")
elif "PASTA_DADOS" not in globals():
    PASTA_DADOS = Path("./dados_cripto")

PASTA_BACKFILL = PASTA_DADOS / "historico_2anos"
PASTA_BACKFILL.mkdir(parents=True, exist_ok=True)

ARQ_BTC = PASTA_BACKFILL / "btc_blocos.csv"
ARQ_BTC_H = PASTA_BACKFILL / "btc_horario.csv"
ARQ_ETH = PASTA_BACKFILL / "eth_blocos.csv"
ARQ_ETH_H = PASTA_BACKFILL / "eth_horario.csv"
ARQ_MERCADO_1H = PASTA_BACKFILL / "mercado_1h.csv"
ARQ_MERCADO_H = PASTA_BACKFILL / "mercado_horario_agregado.csv"
ARQ_FNG = PASTA_BACKFILL / "fear_greed_completo.csv"
ARQ_DATASET = PASTA_BACKFILL / "dataset_2anos_horario.csv"

FIM = pd.Timestamp.now(tz="UTC").floor("h")
INICIO = FIM - pd.Timedelta(days=DIAS)

print(f"Período amplo: {INICIO} -> {FIM} ({DIAS} dias / 2 anos)")
print(f"Pasta de destino: {PASTA_BACKFILL}")

sessao = requests.Session()
sessao.headers.update({"User-Agent": "datathon-cripto-2anos/1.0"})

class RpcError(Exception):
    pass

def req_json(url, params=None, payload=None, tentativas=6, pausa=1.0, timeout=60):
    ultimo = None
    for i in range(tentativas):
        try:
            if payload is None:
                r = sessao.get(url, params=params, timeout=timeout)
            else:
                r = sessao.post(url, json=payload, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                retry_h = r.headers.get("Retry-After")
                try:
                    espera = float(retry_h) if retry_h else pausa * (2 ** i)
                except ValueError:
                    espera = pausa * (2 ** i)
                time.sleep(min(espera, 30))
                continue
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError) as e:
            ultimo = e
            time.sleep(min(pausa * (2 ** i), 30))
    raise RuntimeError(f"Falha em {url}: {ultimo}")

def rpc(url, metodo, params=None):
    resp = req_json(url, payload={"jsonrpc": "2.0", "id": 1, "method": metodo, "params": params or []})
    if "error" in resp:
        raise RpcError(resp["error"])
    return resp["result"]


Período amplo: 2024-11-03 00:00:00+00:00 -> 2026-11-03 00:00:00+00:00 (730 dias / 2 anos)
Pasta de destino: dados_cripto/historico_2anos


## 1. Bitcoin: Coleta Histórica de 2 Anos (`mempool.space`)

Coleta todos os blocos minerados nos últimos 730 dias (~105.000 blocos em lotes de 15). Grava checkpoints a cada 50 chamadas.

In [2]:
URL_BTC = "https://mempool.space/api"

def coletar_btc():
    ts_ini = int(INICIO.timestamp())
    if ARQ_BTC.exists() and not REFAZER:
        antigo = pd.read_csv(ARQ_BTC)
        if not antigo.empty and antigo["timestamp"].min() <= ts_ini:
            print(f"BTC já coberto pelo checkpoint ({len(antigo):,} blocos).")
            return antigo
        linhas = antigo.to_dict("records")
        altura = int(antigo["altura"].min()) - 1
        print(f"Retomando do bloco {altura:,} ({len(linhas):,} blocos já salvos)")
    else:
        linhas = []
        altura = int(req_json(f"{URL_BTC}/blocks/tip/height"))
        print(f"Iniciando do bloco topo {altura:,}")
        
    chamadas = 0
    total_est = DIAS * 144
    t0 = time.time()
    try:
        while altura >= 0:
            lote = req_json(f"{URL_BTC}/v1/blocks/{altura}")
            if not lote:
                break
            for b in lote:
                ex = b.get("extras") or {}
                faixa = ex.get("feeRange") or [None, None]
                linhas.append({
                    "altura": b["height"], "timestamp": b["timestamp"], "tx_count": b.get("tx_count"),
                    "size": b.get("size"), "weight": b.get("weight"),
                    "median_fee": ex.get("medianFee"), "avg_fee_rate": ex.get("avgFeeRate"),
                    "fee_min": faixa[0], "fee_max": faixa[-1], "total_fees": ex.get("totalFees"),
                })
            menor_altura = min(b["height"] for b in lote)
            menor_ts = min(b["timestamp"] for b in lote)
            altura = menor_altura - 1
            chamadas += 1
            
            if chamadas % 50 == 0 or menor_ts < ts_ini:
                pd.DataFrame(linhas).drop_duplicates("altura").to_csv(ARQ_BTC, index=False)
                dt = datetime.fromtimestamp(menor_ts, tz=timezone.utc)
                pct = min(100.0, len(linhas) / total_est * 100)
                dec = time.time() - t0
                print(f"  {chamadas} chamadas | {len(linhas):,} blocos ({pct:.1f}%) | bloco {altura:,} ({dt:%Y-%m-%d}) | {dec:.0f}s")
                
            if menor_ts < ts_ini:
                print(f"Data inicial atingida ({datetime.fromtimestamp(menor_ts, tz=timezone.utc):%Y-%m-%d}).")
                break
            time.sleep(0.15)
    finally:
        if linhas:
            pd.DataFrame(linhas).drop_duplicates("altura").sort_values("altura").to_csv(ARQ_BTC, index=False)
    return pd.read_csv(ARQ_BTC)

df_btc_blocos = coletar_btc()
print(f"Blocos BTC recuperados: {len(df_btc_blocos):,}")

def agregar_btc(df):
    d = df.copy()
    d["hora"] = pd.to_datetime(d["timestamp"], unit="s", utc=True).dt.floor("h")
    d["ocupacao"] = d["weight"] / 4_000_000
    g = d.groupby("hora")
    out = g.agg(
        btc_blocos=("altura", "count"),
        btc_taxa_mediana_sat_vb=("median_fee", "median"),
        btc_taxa_media_sat_vb=("median_fee", "mean"),
        btc_taxa_max_sat_vb=("median_fee", "max"),
        btc_ocupacao_media=("ocupacao", "mean"),
        btc_tx=("tx_count", "sum"),
        btc_taxas_totais_sats=("total_fees", "sum"),
    ).reset_index()
    out.to_csv(ARQ_BTC_H, index=False)
    return out

btc_h = agregar_btc(df_btc_blocos)
print(f"BTC horário gerado: {len(btc_h):,} horas em {ARQ_BTC_H}")
display(btc_h.tail(3))


Retomando do bloco 965,387 (8,925 blocos já salvos)
  50 chamadas | 9,675 blocos (9.2%) | bloco 964,637 (2026-08-29) | 20s
  100 chamadas | 10,425 blocos (9.9%) | bloco 963,887 (2026-08-24) | 40s
  150 chamadas | 11,175 blocos (10.6%) | bloco 963,137 (2026-08-19) | 62s
  200 chamadas | 11,925 blocos (11.3%) | bloco 962,387 (2026-08-14) | 81s
  250 chamadas | 12,675 blocos (12.1%) | bloco 961,637 (2026-08-08) | 103s
  300 chamadas | 13,425 blocos (12.8%) | bloco 960,887 (2026-08-03) | 127s
  350 chamadas | 14,175 blocos (13.5%) | bloco 960,137 (2026-07-29) | 147s
  400 chamadas | 14,925 blocos (14.2%) | bloco 959,387 (2026-07-24) | 166s
  450 chamadas | 15,675 blocos (14.9%) | bloco 958,637 (2026-07-19) | 186s
  500 chamadas | 16,425 blocos (15.6%) | bloco 957,887 (2026-07-13) | 205s
  550 chamadas | 17,175 blocos (16.3%) | bloco 957,137 (2026-07-08) | 223s
  600 chamadas | 17,925 blocos (17.1%) | bloco 956,387 (2026-07-02) | 242s
  650 chamadas | 18,675 blocos (17.8%) | bloco 955,637 (

,hora,btc_blocos,btc_taxa_mediana_sat_vb,btc_taxa_media_sat_vb,btc_taxa_max_sat_vb,btc_ocupacao_media,btc_tx,btc_taxas_totais_sats
17477,2026-11-02 21:00:00+00:00,7,1.200758,1.067949,2.108083,0.998635,37193,11801667
17478,2026-11-02 22:00:00+00:00,5,1.335128,1.262857,1.922038,0.998553,19646,10519035
17479,2026-11-02 23:00:00+00:00,1,0.123907,0.123907,0.123907,0.998185,6027,1290594


## 2. Ethereum: Coleta Histórica de 2 Anos via `eth_feeHistory`

Coleta em lotes de 1.024 blocos (~5.130 chamadas para cobrir 2 anos pós-Merge).

In [3]:
ETH_RPCS = [
    "https://gateway.tenderly.co/public/mainnet",
    "https://eth.drpc.org",
    "https://eth-mainnet.public.blastapi.io"
]

def eth_ts(url, n):
    return int(rpc(url, "eth_getBlockByNumber", [hex(n), False])["timestamp"], 16)

def eth_primeiro_bloco(url, ts, topo):
    ts_topo = eth_ts(url, topo)
    if ts >= ts_topo:
        return topo
    bloco = topo - int((ts_topo - ts) / 12)
    bloco = max(15537393, min(topo, bloco))
    for _ in range(8):
        t = eth_ts(url, bloco)
        diff = ts - t
        if abs(diff) <= 12:
            if t < ts:
                bloco += 1
            return bloco
        passo = int(diff / 12)
        if passo == 0:
            passo = 1 if diff > 0 else -1
        bloco = max(15537393, min(topo, bloco + passo))
    lo = max(15537393, bloco - 100)
    hi = min(topo, bloco + 100)
    while lo < hi:
        mid = (lo + hi) // 2
        if eth_ts(url, mid) < ts:
            lo = mid + 1
        else:
            hi = mid
    return lo

def escolher_rpc_eth():
    for u in ETH_RPCS:
        try:
            topo = int(rpc(u, "eth_blockNumber"), 16)
            return u, topo
        except Exception as e:
            print("RPC indisponível:", u, "-", str(e)[:80])
    raise RuntimeError("Nenhum RPC Ethereum respondeu.")

def coletar_eth():
    ts_ini = int(INICIO.timestamp())
    if ARQ_ETH.exists() and not REFAZER:
        antigo = pd.read_csv(ARQ_ETH)
        if not antigo.empty and antigo["timestamp"].min() <= ts_ini:
            print(f"ETH já coberto pelo checkpoint ({len(antigo):,} blocos).")
            return antigo

    url, topo = escolher_rpc_eth()
    b_ini = eth_primeiro_bloco(url, INICIO.timestamp(), topo)
    b_fim = eth_primeiro_bloco(url, FIM.timestamp(), topo) - 1
    total_blocos = b_fim - b_ini + 1
    print(f"RPC ativo: {url}")
    print(f"Blocos ETH: {b_ini:,} -> {b_fim:,} ({total_blocos:,} blocos, ~{total_blocos // 1024 + 1} chamadas de 1.024 blocos)")
    
    linhas, newest = [], b_fim
    if ARQ_ETH.exists() and not REFAZER:
        antigo = pd.read_csv(ARQ_ETH)
        linhas = antigo.to_dict("records")
        newest = int(antigo["bloco"].min()) - 1
        print(f"Retomando do bloco {newest:,} ({len(linhas):,} blocos já salvos)")
        
    tam, falhas, lotes = 1024, 0, 0
    t0 = time.time()
    try:
        while newest >= b_ini:
            count = min(tam, newest - b_ini + 1)
            try:
                res = rpc(url, "eth_feeHistory", [hex(count), hex(newest), [25, 50, 75]])
                oldest = int(res["oldestBlock"], 16)
                n = len(res["gasUsedRatio"])
                t_start, t_end = eth_ts(url, oldest), eth_ts(url, oldest + n - 1)
            except Exception as e:
                falhas += 1
                if falhas > 25:
                    raise RuntimeError(f"Muitas falhas no feeHistory: {e}")
                if tam > 256:
                    tam //= 2
                else:
                    i = (ETH_RPCS.index(url) + 1) % len(ETH_RPCS) if url in ETH_RPCS else 0
                    url, tam = ETH_RPCS[i], 1024
                    falhas = 0
                    print(f"Trocando RPC para {url}")
                time.sleep(1)
                continue
            falhas = 0
            base = [int(x, 16) / 1e9 for x in res["baseFeePerGas"][:n]]
            for i in range(n):
                ts = t_start + (t_end - t_start) * i / (n - 1) if n > 1 else t_start
                rw = [int(x, 16) / 1e9 for x in res["reward"][i]]
                linhas.append({
                    "bloco": oldest + i, "timestamp": ts, "base_fee_gwei": base[i],
                    "gas_ratio": res["gasUsedRatio"][i],
                    "prio_p25_gwei": rw[0], "prio_p50_gwei": rw[1], "prio_p75_gwei": rw[2]
                })
            newest = oldest - 1
            lotes += 1
            if lotes % 25 == 0 or newest < b_ini:
                pd.DataFrame(linhas).drop_duplicates("bloco").to_csv(ARQ_ETH, index=False)
                restantes = max(newest - b_ini, 0)
                pct = 100 * (1 - restantes / total_blocos)
                dec = time.time() - t0
                print(f"  {lotes} lotes ({pct:.1f}%) | falta ~{restantes:,} blocos | decorrido: {dec:.0f}s")
            time.sleep(0.12)
    finally:
        if linhas:
            pd.DataFrame(linhas).drop_duplicates("bloco").sort_values("bloco").to_csv(ARQ_ETH, index=False)
    return pd.read_csv(ARQ_ETH)

df_eth_blocos = coletar_eth()
print(f"Total de blocos ETH: {len(df_eth_blocos):,}")

def agregar_eth(df):
    d = df.copy()
    d["hora"] = pd.to_datetime(d["timestamp"], unit="s", utc=True).dt.floor("h")
    d["custo_eth"] = ETH_GAS_TRANSFERENCIA * (d["base_fee_gwei"] + d["prio_p50_gwei"]) * 1e-9
    g = d.groupby("hora")
    out = g.agg(
        eth_blocos=("bloco", "count"),
        eth_base_fee_media_gwei=("base_fee_gwei", "mean"),
        eth_base_fee_max_gwei=("base_fee_gwei", "max"),
        eth_gas_ratio_medio=("gas_ratio", "mean"),
        eth_prio_p50_media_gwei=("prio_p50_gwei", "mean"),
        eth_prio_p75_media_gwei=("prio_p75_gwei", "mean"),
        eth_custo_transf_eth=("custo_eth", "mean"),
    ).reset_index()
    out.to_csv(ARQ_ETH_H, index=False)
    return out

eth_h = agregar_eth(df_eth_blocos)
print(f"ETH horário gerado: {len(eth_h):,} horas em {ARQ_ETH_H}")
display(eth_h.tail(3))


RPC ativo: https://gateway.tenderly.co/public/mainnet
Blocos ETH: 21,103,393 -> 26,330,540 (5,227,148 blocos, ~5105 chamadas de 1.024 blocos)
Retomando do bloco 25,899,928 (430,313 blocos já salvos)
  25 lotes (8.7%) | falta ~4,770,935 blocos | decorrido: 28s
  50 lotes (9.2%) | falta ~4,745,335 blocos | decorrido: 56s
  75 lotes (9.7%) | falta ~4,719,735 blocos | decorrido: 84s
  100 lotes (10.2%) | falta ~4,694,135 blocos | decorrido: 112s
  125 lotes (10.7%) | falta ~4,668,535 blocos | decorrido: 141s
  150 lotes (11.2%) | falta ~4,642,935 blocos | decorrido: 170s
  175 lotes (11.7%) | falta ~4,617,335 blocos | decorrido: 199s
  200 lotes (12.2%) | falta ~4,591,735 blocos | decorrido: 228s
  225 lotes (12.6%) | falta ~4,566,135 blocos | decorrido: 257s
  250 lotes (13.1%) | falta ~4,540,535 blocos | decorrido: 286s
  275 lotes (13.6%) | falta ~4,514,935 blocos | decorrido: 314s
  300 lotes (14.1%) | falta ~4,489,335 blocos | decorrido: 343s
  325 lotes (14.6%) | falta ~4,463,735 blo

,hora,eth_blocos,eth_base_fee_media_gwei,eth_base_fee_max_gwei,eth_gas_ratio_medio,eth_prio_p50_media_gwei,eth_prio_p75_media_gwei,eth_custo_transf_eth
17516,2026-11-02 20:00:00+00:00,300,0.137764,0.206868,0.499019,0.047431,0.462910,0.000004
17517,2026-11-02 21:00:00+00:00,300,0.103874,0.153818,0.496879,0.036881,0.894725,0.000003
17518,2026-11-02 22:00:00+00:00,300,0.047708,0.085511,0.495615,0.013143,0.172597,0.000001


## 3. Mercado (Binance 1h) e Sentimento (Fear & Greed) - 2 Anos

Coleta os candles de 1h de BTC e ETH para os 730 dias e integra o Fear & Greed histórico.

In [4]:
BASES_BINANCE = ["https://api.binance.com", "https://data-api.binance.vision"]
SIMBOLOS = {"BTC": "BTCUSDT", "ETH": "ETHUSDT"}
COLS_KLINE = ["abertura_ts", "abertura", "maxima", "minima", "fechamento", "volume",
              "fechamento_ts", "volume_quote", "numero_trades", "volume_taker_base", "volume_taker_quote"]

def klines(simbolo, inicio, fim):
    ini_ms, fim_ms = int(inicio.timestamp() * 1000), int(fim.timestamp() * 1000)
    linhas = []
    while ini_ms < fim_ms:
        dados, erro = None, None
        for base in BASES_BINANCE:
            try:
                dados = req_json(f"{base}/api/v3/klines",
                                 params={"symbol": simbolo, "interval": "1h", "startTime": ini_ms,
                                         "endTime": fim_ms - 1, "limit": 1000}, tentativas=3)
                break
            except RuntimeError as e:
                erro = e
        if dados is None:
            raise erro
        if not dados:
            break
        linhas += [l[:11] for l in dados]
        ini_ms = dados[-1][0] + 1
        if len(dados) < 1000:
            break
        time.sleep(0.15)
    return linhas

def coletar_mercado_1h():
    if ARQ_MERCADO_1H.exists() and not REFAZER:
        df_ex = pd.read_csv(ARQ_MERCADO_1H)
        df_ex["abertura_ts"] = pd.to_datetime(df_ex["abertura_ts"], utc=True)
        if not df_ex.empty and df_ex["abertura_ts".min() if hasattr(df_ex["abertura_ts"], "min") else "abertura_ts"].min() <= INICIO:
            print(f"Mercado 1h carregado do checkpoint ({len(df_ex)} candles).")
            return df_ex

    partes = []
    for ativo, simbolo in SIMBOLOS.items():
        linhas = klines(simbolo, INICIO, FIM)
        df = pd.DataFrame(linhas, columns=COLS_KLINE)
        df["abertura_ts"] = pd.to_datetime(df["abertura_ts"], unit="ms", utc=True)
        df["fechamento_ts"] = pd.to_datetime(df["fechamento_ts"], unit="ms", utc=True)
        for c in COLS_KLINE:
            if c not in ("abertura_ts", "fechamento_ts"):
                df[c] = pd.to_numeric(df[c], errors="coerce")
        df["ativo"] = ativo
        partes.append(df)
        print(f"  {ativo}: {len(df)} candles de 1h")
    out = pd.concat(partes, ignore_index=True).drop_duplicates(["ativo", "abertura_ts"]).sort_values(["ativo", "abertura_ts"]).reset_index(drop=True)
    out.to_csv(ARQ_MERCADO_1H, index=False)
    return out

df_m1h = coletar_mercado_1h()

def agregar_mercado(df_1h):
    h = df_1h.rename(columns={"abertura_ts": "hora"})[["ativo", "hora", "abertura", "maxima", "minima", "fechamento", "volume_quote", "numero_trades"]]
    largo = h.pivot(index="hora", columns="ativo")
    largo.columns = [f"{ativo.lower()}_{campo}" for campo, ativo in largo.columns]
    out = largo.reset_index()
    out.to_csv(ARQ_MERCADO_H, index=False)
    return out

mercado_h = agregar_mercado(df_m1h)
display(mercado_h.tail(3))

# Fear & Greed
arq_local = PASTA_DADOS / "fear_greed_historico.csv"
if arq_local.exists():
    fng = pd.read_csv(arq_local)
    fng["fng_valor"] = pd.to_numeric(fng["value"])
    col_ts = "timestamp_utc" if "timestamp_utc" in fng else "timestamp"
    if col_ts == "timestamp_utc":
        fng["ts"] = pd.to_datetime(fng["timestamp_utc"], utc=True)
    else:
        fng["ts"] = pd.to_datetime(pd.to_numeric(fng["timestamp"]), unit="s", utc=True)
    fng = fng.sort_values("ts")[["ts", "fng_valor", "value_classification"]].reset_index(drop=True)
    fng.to_csv(ARQ_FNG, index=False)
    print(f"Fear & Greed histórico reaproveitado: {len(fng)} dias.")
else:
    resp = req_json("https://api.alternative.me/fng/", params={"limit": 0, "format": "json"})
    fng = pd.DataFrame(resp["data"])
    fng["fng_valor"] = pd.to_numeric(fng["value"])
    fng["ts"] = pd.to_datetime(pd.to_numeric(fng["timestamp"]), unit="s", utc=True)
    fng = fng.sort_values("ts")[["ts", "fng_valor", "value_classification"]].reset_index(drop=True)
    fng.to_csv(ARQ_FNG, index=False)
    print(f"Fear & Greed obtido da API: {len(fng)} dias.")


  BTC: 17520 candles de 1h
  ETH: 17520 candles de 1h


,hora,btc_abertura,eth_abertura,btc_maxima,eth_maxima,btc_minima,eth_minima,btc_fechamento,eth_fechamento,btc_volume_quote,eth_volume_quote,btc_numero_trades,eth_numero_trades
17517,2026-11-02 21:00:00+00:00,76779.72,2248.38,76999.36,2259.22,76723.25,2247.75,76838.01,2258.38,2.656150e+07,1.011299e+07,97385,51638
17518,2026-11-02 22:00:00+00:00,76838.01,2258.38,77011.00,2260.91,76800.00,2249.02,76933.48,2256.87,2.708102e+07,7.624509e+06,52317,43244
17519,2026-11-02 23:00:00+00:00,76933.49,2256.87,76933.49,2257.16,76735.00,2249.25,76816.00,2253.29,2.389224e+07,5.597566e+06,69384,33957


Fear & Greed histórico reaproveitado: 3193 dias.


## 4. Dataset Horário Unificado (2 Anos)

Combina as séries horárias contínuas de BTC, ETH, Mercado e Sentimento e calcula o custo de transação padronizado em USD.

In [5]:
def montar_dataset(mercado_h, btc_h, eth_h, fng):
    def ns(df_, col):
        df_ = df_.copy()
        df_[col] = pd.to_datetime(df_[col], utc=True).astype("datetime64[ns, UTC]")
        return df_

    grade = ns(pd.DataFrame({"hora": pd.date_range(INICIO, FIM - pd.Timedelta(hours=1), freq="h")}), "hora")
    df = grade
    for parte in (mercado_h, btc_h, eth_h):
        if parte is not None and not parte.empty and "hora" in parte.columns:
            df = df.merge(ns(parte, "hora"), on="hora", how="left")

    f = ns(fng.rename(columns={"ts": "hora_fng"}), "hora_fng").sort_values("hora_fng")
    f["fng_valor_lag1d"] = f["fng_valor"].shift(1)
    df = pd.merge_asof(df.sort_values("hora"), f[["hora_fng", "fng_valor", "fng_valor_lag1d"]],
                       left_on="hora", right_on="hora_fng", direction="backward").drop(columns="hora_fng")

    if "btc_taxa_mediana_sat_vb" in df and "btc_fechamento" in df:
        df["btc_custo_usd"] = df["btc_taxa_mediana_sat_vb"] * BTC_TX_VBYTES / 1e8 * df["btc_fechamento"]
    if "eth_custo_transf_eth" in df and "eth_fechamento" in df:
        df["eth_custo_usd"] = df["eth_custo_transf_eth"] * df["eth_fechamento"]

    df["disponivel_em"] = df["hora"] + pd.Timedelta(hours=1)
    df = df.sort_values("hora").reset_index(drop=True)
    df.to_csv(ARQ_DATASET, index=False)
    return df

dataset_2a = montar_dataset(mercado_h, btc_h, eth_h, fng)
print(f"Dataset 2 Anos salvo: {len(dataset_2a):,} linhas x {dataset_2a.shape[1]} colunas em {ARQ_DATASET}")
display(dataset_2a.head(3))
display(dataset_2a.tail(3))


Dataset 2 Anos salvo: 17,520 linhas x 32 colunas em dados_cripto/historico_2anos/dataset_2anos_horario.csv


,hora,btc_abertura,eth_abertura,btc_maxima,eth_maxima,btc_minima,eth_minima,btc_fechamento,eth_fechamento,btc_volume_quote,...,eth_base_fee_max_gwei,eth_gas_ratio_medio,eth_prio_p50_media_gwei,eth_prio_p75_media_gwei,eth_custo_transf_eth,fng_valor,fng_valor_lag1d,btc_custo_usd,eth_custo_usd,disponivel_em
0,2024-11-03 00:00:00+00:00,69374.74,2494.23,69391.00,2496.39,69159.76,2483.01,69159.76,2483.01,3.219343e+07,...,4.406628,0.504156,0.937385,1.750498,0.000090,74,72.0,0.195031,0.223997,2024-11-03 01:00:00+00:00
1,2024-11-03 01:00:00+00:00,69159.77,2483.00,69225.03,2486.78,69000.21,2470.78,69010.13,2471.83,6.056088e+07,...,3.717666,0.502070,0.825882,1.528695,0.000075,74,72.0,0.194609,0.185125,2024-11-03 02:00:00+00:00
2,2024-11-03 02:00:00+00:00,69010.14,2471.82,69028.72,2477.20,67864.16,2426.50,68214.91,2434.60,2.431478e+08,...,8.416222,0.514839,0.985172,1.966675,0.000119,74,72.0,0.192366,0.289622,2024-11-03 03:00:00+00:00


,hora,btc_abertura,eth_abertura,btc_maxima,eth_maxima,btc_minima,eth_minima,btc_fechamento,eth_fechamento,btc_volume_quote,...,eth_base_fee_max_gwei,eth_gas_ratio_medio,eth_prio_p50_media_gwei,eth_prio_p75_media_gwei,eth_custo_transf_eth,fng_valor,fng_valor_lag1d,btc_custo_usd,eth_custo_usd,disponivel_em
17517,2026-11-02 21:00:00+00:00,76779.72,2248.38,76999.36,2259.22,76723.25,2247.75,76838.01,2258.38,2.656150e+07,...,0.153818,0.496879,0.036881,0.894725,0.000003,37,34.0,0.130092,0.006675,2026-11-02 22:00:00+00:00
17518,2026-11-02 22:00:00+00:00,76838.01,2258.38,77011.00,2260.91,76800.00,2249.02,76933.48,2256.87,2.708102e+07,...,0.085511,0.495615,0.013143,0.172597,0.000001,37,34.0,0.144830,0.002884,2026-11-02 23:00:00+00:00
17519,2026-11-02 23:00:00+00:00,76933.49,2256.87,76933.49,2257.16,76735.00,2249.25,76816.00,2253.29,2.389224e+07,...,NaN,NaN,NaN,NaN,NaN,37,34.0,0.013420,NaN,2026-11-03 00:00:00+00:00
